# Write ATST example 5

Read non-plate curves from CSV, define each ATST block, and write one self-contained file.

## Setup

Import the ATST block classes, writer, and data utilities.

In [1]:
from pathlib import Path

import pandas as pd

from ATST import ATSTFile, write_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    Study,
)


## Input data

Read the time-series CSV and curve-keyed layout. No readings are embedded in the notebook.

In [2]:
readings_data = pd.read_csv("example_5_readings.csv")
layout_data = pd.read_csv("example_5_layout.tsv", sep="\t", keep_default_na=False)

readings_data.head()


,Time,ph_1_rep_1,ph_1_rep_2,ph_1_rep_3,ph_2_rep_1,ph_2_rep_2,ph_2_rep_3,ph_3_rep_1,ph_3_rep_2,ph_3_rep_3,...,ph_8_rep_3,ph_9_rep_1,ph_9_rep_2,ph_9_rep_3,ph_10_rep_1,ph_10_rep_2,ph_10_rep_3,isolate_control_1,isolate_control_2,isolate_control_3
0,0,0.172,0.176,0.151,0.168,0.157,0.183,0.179,0.166,0.176,...,0.151,0.180,0.153,0.163,0.139,0.164,0.176,0.187,0.155,0.173
1,40,0.169,0.179,0.170,0.156,0.148,0.189,0.207,0.187,0.189,...,0.148,0.187,0.150,0.174,0.142,0.164,0.168,0.158,0.169,0.185
2,80,0.216,0.233,0.212,0.160,0.141,0.199,0.189,0.217,0.220,...,0.161,0.192,0.151,0.170,0.153,0.159,0.173,0.196,0.212,0.203
3,120,0.296,0.263,0.275,0.173,0.140,0.289,0.280,0.286,0.288,...,0.144,0.178,0.155,0.154,0.138,0.149,0.174,0.269,0.272,0.283
4,160,0.337,0.348,0.354,0.206,0.182,0.342,0.378,0.362,0.376,...,0.142,0.181,0.159,0.181,0.152,0.148,0.173,0.358,0.343,0.350


## FILE_INFO

`FILE_INFO` identifies the output file and ATST format. The file name, format, version, creation date, delimiter, and encoding are required.

In [3]:
file_info = FileMetadata(data={
    "file_name": "example_5.atst.txt",
    "format": "ATST",
    "format_version": "0.1",
    "created_on": "2026-01-01",
    "field_delimiter": "TAB",
    "encoding": "UTF-8",
})


## STUDY

`STUDY` identifies and describes the experiment. A non-empty title and study ID are required.

In [4]:
study = Study(data={
    "title": "Example 5 simulated curve-keyed growth assay",
    "study_id": "ATST_EXAMPLE_5",
    "description": "A non-plate curve-keyed simulated OD600 assay.",
})


## METADATA

`METADATA` records how the curves were produced. This non-plate example describes their simulated source.

In [5]:
metadata = Metadata(data={
    "instrument": "SimulatedReader",
    "plate_type": "non_plate",
    "date_start": "2026-01-01",
    "operator": "OP5",
})


## ASSAY

`ASSAY` identifies the measured signal and its units. Plate format is not required because the layout uses `curve_id`.

In [6]:
assay = Assay(data={
    "readout_type": "absorbance",
    "readout_unit": "od600",
    "time_unit": "min",
})


## ENTITIES

`ENTITIES` defines the biological records referenced by the layout. Each table declares a primary key with unique, non-empty values. These tables should be generated before constructing the final ATST file.

In [7]:
phages = EntityTable(
    name="ENTITIES",
    table_name="PHAGES",
    pk="PHAGE_ID",
    data=pd.read_csv("entities/PHAGES.csv", keep_default_na=False),
)
isolates = EntityTable(
    name="ENTITIES",
    table_name="ISOLATES",
    pk="ISOLATE_ID",
    data=pd.read_csv("entities/ISOLATES.csv", keep_default_na=False),
)
entities = Entities(tables={"PHAGES": phages, "ISOLATES": isolates})


## LAYOUT

`LAYOUT` maps each readings column to its experimental and biological context. Its `well_loc` or `curve_id` key must be unique and match the readings columns.

In [8]:
layout = Layout(data=layout_data)


## READINGS

`READINGS` contains the time series. `Time` is the first column and the remaining column names must match layout keys.

In [9]:
readings = Readings(data=readings_data)


## Write the ATST file

Combine the blocks and write the complete single-file dataset.

In [10]:
atst = ATSTFile(
    file_info=file_info,
    study=study,
    metadata=metadata,
    assay=assay,
    entities=entities,
    layout=layout,
    readings=readings,
)
write_atst(atst, "example_5.atst.txt")


PosixPath('example_5.atst.txt')